In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

chat_model = init_chat_model(model="openai/gpt-oss-120b", model_provider="groq")

d:\Data\Git Projects\langchainupdated\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    name: str = Field(description="The name of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie")

In [3]:
model_with_structured_output = chat_model.with_structured_output(Movie)

model_with_structured_output

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000271A6065550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000271A6065FD0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'name': {'description': 'The name of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The release year of the movie', 'type': 'integer'}, 'genre': {'description': 'The genre of the movie', 'type': 'string'}, 'rating': {'description': 'The

In [5]:
model_with_structured_output.invoke(
    """
    Provide information about the movie Inception?
    """
)

Movie(name='Inception', director='Christopher Nolan', release_year=2010, genre='Science Fiction', rating=8.8)

In [ ]:
class MovieWithEllipsis(BaseModel):
    name: str = Field(...,description="The name of the movie")
    director: str = Field(...,description="The director of the movie")
    release_year: int = Field(...,description="The release year of the movie")
    genre: str = Field(...,description="The genre of the movie")
    rating: float = Field(...,description="The rating of the movie")

model_with_structured_output_and_raw_data = chat_model.with_structured_output(MovieWithEllipsis,include_raw=True)

model_with_structured_output_and_raw_data.invoke("""Provide information about the movie Inception?""")
    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants info about movie Inception. We have function MovieWithEllipsis that likely returns movie info given director, genre, name, rating, release_year. We need to fill fields. Inception director Christopher Nolan, genre sci-fi thriller, rating maybe 8.8? Could be IMDb rating 8.8. Release year 2010. Provide info. Use function.', 'tool_calls': [{'id': 'fc_656f22af-149b-49f9-aec9-718b6d069c50', 'function': {'arguments': '{"director":"Christopher Nolan","genre":"Science Fiction / Thriller","name":"Inception","rating":8.8,"release_year":2010}', 'name': 'MovieWithEllipsis'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 171, 'total_tokens': 315, 'completion_time': 0.309239201, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.007161998, 'prompt_tokens_details': None, 'queue_time': 0.055455882, 'total_time': 0.316401199}, 'model_name': 'openai

In [9]:
## Nested Structure

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    actors: list[Actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in crores")

model_with_structure = chat_model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details of the movie Inception"
)

print(response)

title='Inception' year=2010 actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')] genre=['Action', 'Adventure', 'Sci-Fi', 'Thriller'] budget=160000000.0


In [ ]:
# TypedDict

from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of release"]
    actors: Annotated[list[str], ..., "The actors in the movie"]
    genre: Annotated[list[str], ..., "The genre of the movie"]
    budget: Annotated[float | None, ..., "The budget of the movie"]

model_with_structure_typedDict = chat_model.with_structured_output(MovieDict)

response_typedDict = model_with_structure_typedDict.invoke(
    "Provide details of the movie Inception"
)

print(response_typedDict)


{'actors': ['Leonardo DiCaprio', 'Joseph Gordon-Levitt', 'Elliot Page', 'Tom Hardy', 'Ken Watanabe', 'Marion Cotillard', 'Michael Caine'], 'budget': 160000000, 'genre': ['Science Fiction', 'Action', 'Thriller'], 'title': 'Inception', 'year': 2010}


In [11]:
chat_model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [ ]:
# Data Class

# 1. BaseModel Pydantic
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    chat_model,
    response_format=ContactInfo # auto-selects ProviderStrategy
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Extract the contact information from: John Doe, john@example.com, 1234567890"}]}
)

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract the contact information from: John Doe, john@example.com, 1234567890', additional_kwargs={}, response_metadata={}, id='87f13dae-2b39-45df-a1eb-89bc79c1e6e2'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"1234567890"}', additional_kwargs={'reasoning_content': 'The user asks to extract contact info from a string. The system says to output only final JSON object adhering to ContactInfo schema. Must include name, email, phone. Provide compact JSON. So output {"name":"John Doe","email":"john@example.com","phone":"1234567890"}.\n\nCheck schema required fields present. Yes.\n\nMake sure compact (no whitespace). Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 111, 'prompt_tokens': 235, 'total_tokens': 346, 'completion_time': 0.233494954, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.009624561, 'prompt_tokens_details': None, 'queue_time': 0.054877109, 'total_time': 0.2

In [ ]:
# 2. TypedDict

class ContactInfo(TypedDict):
    """Contact information for a person"""
    name: str # The name of the person 
    email: str # The email of the person 
    phone: str # The phone number of the person 

agent = create_agent(
    chat_model,
    response_format=ContactInfo # auto-selects ProviderStrategy
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Extract the contact information from: John Doe, john@example.com, 1234567890"}]}
)

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract the contact information from: John Doe, john@example.com, 1234567890', additional_kwargs={}, response_metadata={}, id='866ddfad-0b97-4535-95cf-be8e5e39d6a9'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"1234567890"}', additional_kwargs={'reasoning_content': 'The user wants extraction of contact info. The system says we must output only final JSON object adhering to ContactInfo schema. Include name, email, phone. So output {"name":"John Doe","email":"john@example.com","phone":"1234567890"} Ensure compact JSON.'}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 210, 'total_tokens': 298, 'completion_time': 0.19178194, 'completion_tokens_details': {'reasoning_tokens': 57}, 'prompt_time': 0.009584517, 'prompt_tokens_details': None, 'queue_time': 0.054940883, 'total_time': 0.201366457}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8a618bed98', 'service_tier': 'on_demand', 

In [ ]:
# 3. Data Class

from dataclasses import dataclass

@dataclass
class ContactInfo:
    name: str # The name of the person 
    email: str # The email of the person 
    phone: str # The phone number of the person 

agent = create_agent(
    chat_model,
    response_format=ContactInfo # auto-selects ProviderStrategy
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Extract the contact information from: John Doe, john@example.com, 1234567890"}]}
)

print(result)
print(result['structured_response'])

{'messages': [HumanMessage(content='Extract the contact information from: John Doe, john@example.com, 1234567890', additional_kwargs={}, response_metadata={}, id='b7a864fe-a84a-454f-b66d-3a87572025f0'), AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"1234567890"}', additional_kwargs={'reasoning_content': 'We need to output JSON matching ContactInfo schema: name, email, phone strings. Provide compact JSON. So:\n\n{"name":"John Doe","email":"john@example.com","phone":"1234567890"}'}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 202, 'total_tokens': 273, 'completion_time': 0.148449722, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.009409764, 'prompt_tokens_details': None, 'queue_time': 0.054264855, 'total_time': 0.157859486}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_r